# Difference between Genbank and NCBI Virus data

Question: What data is in Genbank that is not in NCBI Virus?

In [ ]:
# Housekeeping

import os
import glob 
import pandas as pd
import xml.etree.ElementTree as ET
import requests
import time
import numpy as np
import dateutil 
from datetime import datetime
from collections import defaultdict 
import importlib
import utils  
importlib.reload(utils)
from utils import * 

In [2]:
# Paths

# home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
downloads_ncbi_virus = home + "NCBI_Virus_Downloads/"
downloads_genbank = home + "Genbank_Downloads/05-13-2025/"



In [11]:
os.chdir(downloads_ncbi_virus)
metadata_ncbi_virus = pd.read_csv("sequences_all_05-13-2025.csv")
metadata_ncbi_virus_names = metadata_ncbi_virus["GenBank_Title"].values
metadata_ncbi_virus_accessions = set(metadata_ncbi_virus["Accession"].values)
# print(metadata_ncbi_virus["GenBank_Title"].values[:50])
# print(metadata_ncbi_virus.columns)
os.chdir(downloads_genbank)
metadata_genbank_names = []
metadata_genbank_accessions = set()
for_csv = {}
with open("nuccore_result.txt") as f:
    lines = f.readlines()
    for i, line in enumerate(lines):
        if "Influenza A virus" in line: # Title line
            name = line.split(". ")[1]
            metadata_genbank_names.append(name)
            # Accession number is two lines after
            accession = lines[i + 2].split(" GI:")[0]
            gi = lines[i + 2].split(" GI:")[1][:-1]
            metadata_genbank_accessions.add(accession.strip())
            for_csv[accession] = [name, gi]
    f.close()

C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\3\ipykernel_26368\1435493951.py:2: DtypeWarning: Columns (4,17,25) have mixed types. Specify dtype option on import or set low_memory=False.
  metadata_ncbi_virus = pd.read_csv("sequences_all_05-13-2025.csv")


In [12]:
# metadata_genbank_names_set = set(metadata_genbank_names)

print("Number of Genbank H5N1 sequences as of 5/13/2025:", len(metadata_genbank_accessions))

# metadata_ncbi_virus_names_set = set(metadata_ncbi_virus_names)
genbank_only = metadata_genbank_accessions - metadata_ncbi_virus_accessions

# genbank_only = []
# for genbank_name in metadata_genbank_names:
#     if genbank_name not in metadata_ncbi_virus_names:
#         genbank_only.append(genbank_name)

print("Number of NCBI Virus H5N1 sequences: ", len(metadata_ncbi_virus_accessions))

print("Number of Genbank H5N1 sequences not found in NCBI Virus:", len(genbank_only))

Number of Genbank H5N1 sequences as of 5/13/2025: 122335
Number of NCBI Virus H5N1 sequences:  116373
Number of Genbank H5N1 sequences not found in NCBI Virus: 5982


In [13]:
for_csv_df = pd.DataFrame.from_dict(for_csv, orient="index", columns=["Name", "GI"])
for_csv_df["Accession"] = for_csv_df.index
for_csv_df = for_csv_df.reset_index(drop=True)

print(for_csv_df)

                                                     Name        GI  \
0       Influenza A virus (A/Pheasant/Hong Kong/FY155/...  28849718   
1       Influenza A virus (A/Chicken/Hong Kong/FY150/0...  28849716   
2       Influenza A virus (A/Chicken/Hong Kong/YU562/0...  28849712   
3       Influenza A virus (A/Pheasant/Hong Kong/FY155/...  28849666   
4       Influenza A virus (A/Chicken/Hong Kong/FY150/0...  28849664   
...                                                   ...       ...   
122330  Influenza A virus genomic RNA for polymerase P...  18074810   
122331  Influenza A virus genomic RNA for polymerase P...  18074808   
122332  Influenza A virus genomic RNA for polymerase P...  18074806   
122333  Influenza A virus genomic RNA for polymerase P...  18074804   
122334  Influenza A virus genomic RNA for polymerase P...  18074802   

         Accession  
0       AF509199.1  
1       AF509198.1  
2       AF509196.1  
3       AF509173.1  
4       AF509172.1  
...            ...  


In [14]:
genbank_only_df = for_csv_df[for_csv_df['Accession'].isin(genbank_only)]
print(genbank_only_df)
genbank_only_df.to_csv("genbank_only.csv")

                                                     Name        GI  \
0       Influenza A virus (A/Pheasant/Hong Kong/FY155/...  28849718   
1       Influenza A virus (A/Chicken/Hong Kong/FY150/0...  28849716   
2       Influenza A virus (A/Chicken/Hong Kong/YU562/0...  28849712   
3       Influenza A virus (A/Pheasant/Hong Kong/FY155/...  28849666   
4       Influenza A virus (A/Chicken/Hong Kong/FY150/0...  28849664   
...                                                   ...       ...   
122330  Influenza A virus genomic RNA for polymerase P...  18074810   
122331  Influenza A virus genomic RNA for polymerase P...  18074808   
122332  Influenza A virus genomic RNA for polymerase P...  18074806   
122333  Influenza A virus genomic RNA for polymerase P...  18074804   
122334  Influenza A virus genomic RNA for polymerase P...  18074802   

         Accession  
0       AF509199.1  
1       AF509198.1  
2       AF509196.1  
3       AF509173.1  
4       AF509172.1  
...            ...  


In [ ]:
genbank_h5n1_only = genbank_only_df[genbank_only_df["Name"].apply(lambda x: "H5N1" in x)]
genbank_h5n1_only["host"] = genbank_h5n1_only["Name"].apply(lambda x: x.split("/")[1] if len(x.split("/")) > 1 else x)
genbank_h5n1_only["geo_location"] = genbank_h5n1_only["Name"].apply(lambda x: x.split("/")[2] if len(x.split("/")) > 1 else x)
genbank_h5n1_only["year"] = genbank_h5n1_only["Name"].apply(lambda x: x.split("/")[-1][:4] if len(x.split("/")[-1]) > 4 else x)

print(genbank_h5n1_only)
# genbank_h5n1_only.to_csv("genbank_h5n1_only.csv")

                                                     Name          GI  \
0       Influenza A virus (A/Pheasant/Hong Kong/FY155/...    28849718   
1       Influenza A virus (A/Chicken/Hong Kong/FY150/0...    28849716   
2       Influenza A virus (A/Chicken/Hong Kong/YU562/0...    28849712   
3       Influenza A virus (A/Pheasant/Hong Kong/FY155/...    28849666   
4       Influenza A virus (A/Chicken/Hong Kong/FY150/0...    28849664   
...                                                   ...         ...   
70306   Influenza A virus (A/wood duck/Ohio/23OS0170/2...  2972151673   
70309   Influenza A virus (A/wood duck/Ohio/23OS0170/2...  2972151664   
70310   Influenza A virus (A/wood duck/Ohio/23OS0170/2...  2972151662   
70314   Influenza A virus (A/British_Columbia/PHL-2032...  2972099020   
118705  Synthetic construct Influenza A virus (A/whoop...   262475079   

         Accession              host          geo_location  year  
0       AF509199.1          Pheasant             Hong Ko

C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\3\ipykernel_26368\2331539827.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  genbank_h5n1_only["host"] = genbank_h5n1_only["Name"].apply(lambda x: x.split("/")[1] if len(x.split("/")) > 1 else x)
C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\3\ipykernel_26368\2331539827.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  genbank_h5n1_only["geo_location"] = genbank_h5n1_only["Name"].apply(lambda x: x.split("/")[2] if len(x.split("/")) > 1 else x)
C:\Users\m

In [21]:
#assemble the esearch URL
query = "2972151673"
base = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/';
url = base + "esearch.fcgi?db=nucleotide&term=" + query + "&usehistory=y";

#post the esearch URL
output = requests.get(url)

#parse WebEnv, QueryKey and Count (# records retrieved)
xml = output.content
root = ET.fromstring(xml)
query_key = root.find(".//QueryKey").text
web_env = root.find(".//WebEnv").text

nucleotide_url = base + "esummary.fcgi?db=nuccore&query_key=" + query_key + "&WebEnv=" + web_env + "&version=2.0&api_key=2cbaf77ac9ec5ae7844ea350076ae6d56809"

print(nucleotide_url)

output = requests.get(nucleotide_url) 
xml = output.content
root = ET.fromstring(xml)

#open output file for writing
# open(OUT, ">chimp.fna") || die "Can't open file!\n";

#retrieve data in batches of 500
# $retmax = 500;
# for ($retstart = 0; $retstart < $count; $retstart += $retmax) {
#         $efetch_url = $base ."efetch.fcgi?db=nucleotide&WebEnv=$web";
#         $efetch_url .= "&query_key=$key&retstart=$retstart";
#         $efetch_url .= "&retmax=$retmax&rettype=fasta&retmode=text";
#         $efetch_out = get($efetch_url);
#         print OUT "$efetch_out";
# }
# close OUT;

https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=nuccore&query_key=1&WebEnv=MCID_68236bec387591f5ce07d901&version=2.0&api_key=2cbaf77ac9ec5ae7844ea350076ae6d56809
